# MX-mantissatruncatie: E2M3 naar E2M1

Deze notebook **leest opgeslagen evaluaties**. Geen training, downloads, modelcalls of BatchNorm-herkalibratie.

**H:** opgeslagen E2M3 met hoge BN-bank (context). **A:** oorspronkelijke directe E2M1-export.
**B:** naar E2M1 afronden vanuit de opgeslagen E2M3-elementen, met ongewijzigde schaal.
**C:** twee mantissabits verwijderen uit die E2M3-codes, zonder afronding of nieuwe schaal.

A, B en C gebruiken dezelfde eerder geleerde **lage BN-bank** en A32. C is de eigenlijke truncatieproef;
B scheidt bronkwantisatie van het verschil tussen afronden en trunceren. Beide checkpoints blijven vast.
De hoogste broncodes blijven behouden: de lage modus is geen overschrijving van het bronmodel.


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

RUN_DIR = Path(".")  # RUN_DIRECTORY
report = json.loads((RUN_DIR / "run_report.json").read_text(encoding="utf-8"))
if report.get("synthetic_fixture_only"):
    display(Markdown("## SYNTHETISCHE SOFTWARETEST - GEEN ONDERZOEKSRESULTATEN"))
assert report["status"] == "PASS", "Geen geslaagde evaluatie; bekijk het fout-/waarschuwingsrapport."
assert report.get("optimizer_steps") == 0 and report.get("batchnorm_recalibration") is False
metrics = pd.read_csv(RUN_DIR / "metrics.csv")
pairs = pd.read_csv(RUN_DIR / "paired_metrics.csv")
weights = pd.read_csv(RUN_DIR / "weight_pair_metrics.csv")
counts = pd.read_csv(RUN_DIR / "layer_code_metrics.csv")
expected_paths = {"H_source_E2M3", "A_direct_E2M1", "B_round_from_E2M3", "C_truncate_E2M3"}
assert set(metrics.path) == expected_paths
assert not metrics.duplicated(["role", "path"]).any()
assert len(metrics) == 4 * metrics.role.nunique()
assert np.isfinite(metrics[["top1_percent", "hard_cross_entropy"]]).all().all()
assert np.allclose(metrics.top1_percent, 100 * metrics.correct1 / metrics.n, rtol=0, atol=1e-10)
assert (metrics.activation_bits == 32).all()
assert (metrics.loc[metrics.path != "H_source_E2M3", "bn_bank_key"] == 4).all()
print("Gelezen map:", RUN_DIR)
print("Status:", report["status"], "| waarschuwingen:", len(report.get("warnings", [])))


## 1. Testaccuracy op dezelfde beelden

Vergelijk **C met A** bij hetzelfde checkpoint. De hoge E2M3-modus gebruikt haar eigen BN-bank en is context,
geen vervanging voor de directe E2M1-controle. Het beste checkpoint werd al tijdens de training gekozen;
epoch 100 blijft het vaste eindpunt. De tabel kiest geen nieuw best checkpoint.


In [ ]:
display(metrics[["role", "epoch", "path", "correct1", "n", "top1_percent", "hard_cross_entropy"]]
        .style.format({"top1_percent": "{:.4f}%", "hard_cross_entropy": "{:.6f}"}))

for role, frame in metrics.groupby("role", sort=False):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(frame.path.str[0], frame.top1_percent)
    ax.set(ylim=(0, 100), ylabel="Testaccuracy (%)", xlabel="Evaluatiepad",
           title=f"{role} - epoch {int(frame.epoch.iloc[0])}")
    for x, value in enumerate(frame.top1_percent):
        ax.annotate(f"{value:.3f}%", (x, value), xytext=(0, 4), textcoords="offset points", ha="center")
    fig.tight_layout()
    plt.show()


## 2. Gepaarde voorspellingen

Een klein netto verschil betekent niet dat de voorspellingen gelijk zijn. We tellen daarom fout-naar-correct,
correct-naar-fout en fout-naar-een-andere-fout. Positieve `accuracy_difference_pp` betekent dat het eerste pad in
`comparison` beter scoort. Er wordt geen statistische gelijkwaardigheid geclaimd uit twee checkpoints van een seed.


In [ ]:
display(pairs[["role", "comparison", "accuracy_difference_pp", "prediction_changes", "wrong_to_correct",
               "correct_to_wrong", "wrong_to_other_wrong", "net_correct_change"]]
        .style.format({"accuracy_difference_pp": "{:+.4f}"}))
assert (pairs.net_correct_change == pairs.wrong_to_correct - pairs.correct_to_wrong).all()
assert (pairs.prediction_changes == pairs.wrong_to_correct + pairs.correct_to_wrong + pairs.wrong_to_other_wrong).all()


## 3. Controle van de opgeslagen individuele uitvoer

De feedbackbundel bevat nu ook labels, indices, voorspellingen, logits en cross-entropy per beeld. Dat maakt
latere gepaarde analyse mogelijk. Er staan geen ruwe beelden, checkpointgewichten of FP32-mastergewichten in.


In [ ]:
verified = []
for role, details in report["roles"].items():
    file = RUN_DIR / details["per_image_file"]
    assert hashlib.sha256(file.read_bytes()).hexdigest() == details["per_image_sha256"]
    with np.load(file, allow_pickle=False) as z:
        labels, indices = z["labels"], z["indices"]
        assert np.array_equal(indices, np.arange(len(labels)))
        for path in expected_paths:
            row = metrics[(metrics.role == role) & (metrics.path == path)].iloc[0]
            predictions = z[path + "__predictions"]
            logits = z[path + "__logits"]
            losses = z[path + "__per_image_ce"]
            assert logits.shape == (len(labels), 10) and np.isfinite(logits).all()
            assert np.isfinite(losses).all()
            assert int((predictions == labels).sum()) == int(row.correct1)
            assert np.isclose(losses.astype(np.float64).mean(), row.hard_cross_entropy, rtol=0, atol=1e-12)
        verified.append({"role": role, "n": len(labels), "raw_outputs_verified": True})
display(pd.DataFrame(verified))


## 4. Code- en gewichtsverschillen per laag

De numerieke afwijkingen zijn **tussen de gedecodeerde evaluatiepaden**, niet ten opzichte van de ontbrekende
FP32-master. Padding telt niet mee als echt gewicht. Voor C moeten teken, lokaal exponentveld en gedeelde schaal
behouden blijven, ook bij negatieve waarden en lokale subnormals.


In [ ]:
display(counts[["role", "layer", "n_weights", "blocks", "padding_slots", "nonzero_dropped_suffix_count",
                "C_new_zero_vs_high", "sign_exponent_scale_preserved_C"]])
display(weights[["role", "layer", "comparison", "code_differences", "numeric_differences", "mae", "max_abs_difference"]])
assert counts.sign_exponent_scale_preserved_C.all()
for role, frame in weights.groupby("role", sort=False):
    fig, ax = plt.subplots(figsize=(8, 4))
    for comparison, rows in frame.groupby("comparison", sort=False):
        ax.plot(rows.layer, rows.mae, marker="o", label=comparison)
    ax.set(title=role + " - gemiddelde absolute gewichtsverschillen", ylabel="MAE (gedecodeerde waarden)", xlabel="Laag")
    ax.legend()
    fig.tight_layout()
    plt.show()


## 5. Opslag en reconstructie

Een volledig blok gebruikt 24 codebytes + 1 schaalbyte in E2M3, en 16 + 1 in E2M1. De tabel telt de padding mee.
Aanvullende modelstaat en serialisatieheaders staan hier niet bij. Om terug naar zes bit te kunnen schakelen blijft
de hoogste bron behouden. De afgeleide vierbitsbestanden in deze diagnose zijn auditresultaten, geen claim dat
alle uitvoeringsmodi permanent tegelijk moeten worden opgeslagen. FP32-rekenbuffers zijn geen native MX-kernel.


In [ ]:
display(counts.groupby("role", sort=False)[["n_weights", "blocks", "padding_slots", "high_code_and_scale_bytes",
                                         "low_code_and_scale_bytes"]].sum())
for role, details in report["roles"].items():
    assert details["all_source_scales_equal"] and details["same_auxiliary_state"]
    assert details["all_model_and_bn_states_unchanged"] and details["optimizer_steps"] == 0
    assert details["six_FP32_master_parameters_absent_all_paths"]
print("Geen gewijzigde gewichten/BatchNorm-statistieken tijdens evaluatie.")


## 6. Reproductie van de oorspronkelijke directe scores en beperkingen

A en H zijn de directe exports van de reeds getrainde checkpoints. Hun nieuwe testscores worden met de
opgeslagen trainingsscores vergeleken. Een afwijking wordt zichtbaar gerapporteerd; we passen niets aan om die
weg te werken. CPU-export, GPU-afronding en niet-volledig-deterministische uitvoering kunnen een rol spelen,
maar vormen zonder verdere controle geen bewezen verklaring. De nieuwe gepaarde A-C-vergelijking is leidend.


In [ ]:
reproduction_file = RUN_DIR / "historical_reproduction.csv"
if reproduction_file.is_file():
    display(pd.read_csv(reproduction_file))
else:
    print("Geen historische referenties in deze synthetische test.")
if report.get("warnings"):
    display(pd.DataFrame(report["warnings"]))
else:
    print("Geen waarschuwingen geregistreerd.")
print("Gemeten sessieduur (geen hardwarebenchmark):", round(report.get("session_wall_seconds", 0), 2), "s")
for limitation in report.get("limitations", []):
    print("-", limitation)


## Interpretatie

Een accuracyverschil C-A meet het effect van deze specifieke truncatie-ingreep op dit vaste, voor **directe
kwantisatie** getrainde model. A-B maakt het effect van eerst een E2M3-bron vormen zichtbaar; B-C vergelijkt
afronden en trunceren vanuit diezelfde bron. De verschillen in totale accuracy tellen rekenkundig op (C-A = C-B + B-A), maar de richting van de
voorspellingsveranderingen kan per beeld verschillen. Dat bewijst geen onafhankelijke causale bijdragen.

Geen training, geen BatchNorm-herkalibratie, geen nieuwe selectie. Een slechte truncatiescore laat de technische
status PASS ongemoeid wanneer de implementatiecontroles slagen: die score is juist een mogelijk onderzoeksresultaat.
